In [15]:
# OptimumFilterCUDA (CuPy/cuFFT version) — same API, no Numba required
import numpy as np

try:
    import cupy as cp
    from cupyx.scipy.fft import fft as cufft
    try:
        # CuPy 13+ has this
        from cupyx.scipy.signal import sliding_window_view as cp_sliding_window_view
    except Exception:
        # fallback to core stride_tricks
        cp_sliding_window_view = cp.lib.stride_tricks.sliding_window_view
    CUPY_OK = cp.is_available()
except Exception:
    cp = None
    CUPY_OK = False


class OptimumFilterCUDA:
    """
    GPU-accelerated sliding-fit using CuPy (cuFFT batched).
    - Same API as your Numba version (set_template, set_noise_psd, sliding_fit).
    - If CuPy/GPU is unavailable, it seamlessly falls back to NumPy (CPU).

    The sliding_fit is implemented via overlapping-window batches + cuFFT,
    which is typically faster and avoids sliding-DFT recurrences.
    """

    def __init__(self, template, noise_psd, sampling_frequency, threads_per_block=None):
        # threads_per_block kept for interface compatibility (unused)
        self._sampling_frequency = float(sampling_frequency)
        self.set_template(template)
        self.set_noise_psd(noise_psd)

    # -------- public API (unchanged) --------
    def set_template(self, template):
        self._template = np.asarray(template, dtype=np.float64)
        self._length = self._template.size
        self._rebuild_state()

    def set_noise_psd(self, noise_psd):
        self._noise_psd = np.asarray(noise_psd, dtype=np.float64)
        self._rebuild_state()

    def sliding_fit(self, trace_long, hop=1, reanchor_every=None, tile_windows=8192):
        """
        Sliding fit over a long trace using batched FFT windows.
        Args:
            trace_long: 1D array (length L)
            hop: hop size between windows (int >= 1)
            reanchor_every: kept for API compatibility (not used in batched FFT mode)
            tile_windows: number of windows per GPU tile to bound VRAM
        Returns:
            amps, chisqs: 1D arrays length = number of windows
        """
        x = np.asarray(trace_long, dtype=np.float64)
        L, N, fs = int(x.size), self._length, self._sampling_frequency
        if L < N:
            raise ValueError("Trace shorter than template length.")
        if hop <= 0:
            raise ValueError("hop must be a positive integer.")

        num_windows = 1 + (L - N) // hop
        amps_host = np.empty(num_windows, dtype=np.float64)
        chis_host = np.empty(num_windows, dtype=np.float64)

        if CUPY_OK:
            # ---- GPU path (CuPy) ----
            xg = cp.asarray(x)
            Fg = self._F_gpu
            Sg = self._S_unf_gpu

            # Create one sliding view and index per tile (saves memory & time)
            swv = cp_sliding_window_view(xg, N)  # shape: (L-N+1, N)

            w0 = 0
            while w0 < num_windows:
                w1 = min(w0 + tile_windows, num_windows)
                starts = cp.arange(w0, w1) * hop
                windows = swv[starts]                        # (tile, N)

                X = cufft(windows, axis=-1) / fs             # (tile, N)
                XF = X * Fg[None, :]                         # (tile, N)

                # amp = Re(sum(XF)) * fs / N
                amps_tile = cp.real(XF.sum(axis=-1)) * fs / N

                # chi0 = sum(|X|^2 / S_unf) * fs / N
                mod2 = (X.real * X.real + X.imag * X.imag)
                chi0_tile = (mod2 / Sg[None, :]).sum(axis=-1) * fs / N
                chis_tile = (chi0_tile - amps_tile**2 * self._kernel_normalization) / (N - 2)

                amps_host[w0:w1] = cp.asnumpy(amps_tile)
                chis_host[w0:w1] = cp.asnumpy(chis_tile)
                w0 = w1

        else:
            # ---- CPU fallback (NumPy) ----
            from numpy.lib.stride_tricks import sliding_window_view as np_sliding_window_view
            F = self._F_cpu
            S = self._S_unf_cpu

            swv = np_sliding_window_view(x, N)  # (L-N+1, N)
            w0 = 0
            while w0 < num_windows:
                w1 = min(w0 + tile_windows, num_windows)
                starts = np.arange(w0, w1) * hop
                windows = swv[starts]                         # (tile, N)

                X = np.fft.fft(windows, axis=-1) / fs
                XF = X * F[None, :]
                amps_tile = np.real(XF.sum(axis=-1)) * fs / N

                mod2 = (X.real * X.real + X.imag * X.imag)
                chi0_tile = (mod2 / S[None, :]).sum(axis=-1) * fs / N
                chis_tile = (chi0_tile - amps_tile**2 * self._kernel_normalization) / (N - 2)

                amps_host[w0:w1] = amps_tile
                chis_host[w0:w1] = chis_tile
                w0 = w1

        return amps_host, chis_host

    # -------- internals --------
    def _rebuild_state(self):
        """Build PSD-unfolded, kernel FFT, and GPU caches."""
        if not hasattr(self, "_template") or not hasattr(self, "_noise_psd"):
            return
        N = self._length
        fs = self._sampling_frequency

        # Unfold PSD exactly like your original code
        if N % 2 == 0:
            S_unf = np.concatenate((
                [np.inf],
                self._noise_psd[1:-1] / 2,
                [self._noise_psd[-1]],
                self._noise_psd[-2:0:-1] / 2
            ))
        else:
            S_unf = np.concatenate((
                [np.inf],
                self._noise_psd[1:] / 2,
                self._noise_psd[-1:0:-1] / 2
            ))
        self._S_unf_cpu = S_unf

        T_fft = np.fft.fft(self._template) / fs
        kernel_fft = np.conjugate(T_fft) / S_unf
        self._kernel_normalization = float(np.real(np.vdot(T_fft, kernel_fft)) * fs / N)
        self._F_cpu = (kernel_fft / self._kernel_normalization).astype(np.complex128)

        # GPU copies
        if CUPY_OK:
            self._F_gpu = cp.asarray(self._F_cpu)
            self._S_unf_gpu = cp.asarray(self._S_unf_cpu)


In [16]:
import numpy as np
import matplotlib.pyplot as plt


sampling_frequency = 3906250
vac_template = np.load("/home/dwong/DELight_mtr/trigger_study/archive/wk15/templates/vac_ch_template.npy")

noise_psd = np.load("/home/dwong/DELight_mtr/templates/noise_psd_from_MMC.npy")
vac_of = OptimumFilterCUDA(vac_template, noise_psd, sampling_frequency)


data = np.load("separate_trace.npz")
signal = data["signal"]
noise = data["noise"]
quantized = data["quantized"] 
unquantized = data["unquantized"] 

In [17]:
amps_signal, chisq_signal = vac_of.sliding_fit(signal, hop=1)


In [6]:
# --- TURN MVC OFF BEFORE ANY numba/cuda IMPORTS ---
import os, sys

# 1) Ensure env var is unset for this process
os.environ.pop("NUMBA_CUDA_ENABLE_MINOR_VERSION_COMPATIBILITY", None)

# 2) Ensure Numba config flag is off
from numba import config
config.CUDA_ENABLE_MINOR_VERSION_COMPATIBILITY = 0

# 3) If any numba.cuda modules were imported by earlier cells (or autoreload),
#    drop them so a clean import will pick up the settings above.
for m in list(sys.modules):
    if m.startswith("numba.cuda"):
        del sys.modules[m]

print("MVC env var:", os.environ.get("NUMBA_CUDA_ENABLE_MINOR_VERSION_COMPATIBILITY"))
print("MVC config:", config.CUDA_ENABLE_MINOR_VERSION_COMPATIBILITY)


MVC env var: None
MVC config: 0


In [7]:
# Check that we're really using nvrtc 12.0 (PTX 8.0)
import ctypes, ctypes.util
p = ctypes.util.find_library("nvrtc")
print("nvrtc path:", p)
nvrtc = ctypes.CDLL(p)
maj, min_ = ctypes.c_int(), ctypes.c_int()
nvrtc.nvrtcVersion(ctypes.byref(maj), ctypes.byref(min_))
print("nvrtc version:", f"{maj.value}.{min_.value}")

# Now import CUDA (MVC should NOT be attempted)
from numba import cuda
print("CUDA available:", cuda.is_available())

# Optional: see driver version that Numba detects
from numba.cuda.cudadrv import driver
print("Driver version:", driver.get_version())  # e.g., (12, 0)


nvrtc path: libnvrtc.so.12
nvrtc version: 12.0
CUDA available: True
Driver version: (12, 0)


In [8]:
import numpy as np

@cuda.jit
def add1(x):
    i = cuda.grid(1)
    if i < x.size:
        x[i] += 1

a = np.zeros(128, np.float32)
d = cuda.to_device(a)
add1[1, 128](d)
print(d.copy_to_host()[:8])  # expect all ones


/home/dwong/anaconda3/envs/ofgpu/lib/python3.12/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


AssertionError: key already in dictionary: <class 'numba.core.target_extension.CUDA'>

In [9]:
import os, sys, ctypes

# 1) Turn MVC OFF (so Numba won't ask for cubinlinker/ptxcompiler)
os.environ.pop("NUMBA_CUDA_ENABLE_MINOR_VERSION_COMPATIBILITY", None)
from numba import config
config.CUDA_ENABLE_MINOR_VERSION_COMPATIBILITY = 0

# 2) Remove system CUDA from loader/search paths for THIS kernel process
def _strip_cuda(var):
    paths = [p for p in os.environ.get(var, "").split(":") if p and not p.startswith("/usr/local/cuda")]
    os.environ[var] = ":".join(paths)
for v in ("PATH", "LD_LIBRARY_PATH"):
    _strip_cuda(v)

# 3) Force-load conda’s NVRTC 12.0 *globally* before importing numba.cuda
#    (adjust this path if your conda prefix differs)
nvrtc_path = "/home/dwong/anaconda3/lib/libnvrtc.so.12"
hdl = ctypes.CDLL(nvrtc_path, mode=ctypes.RTLD_GLOBAL)

# 4) Verify we're really using NVRTC 12.0
maj = ctypes.c_int(); min_ = ctypes.c_int()
hdl.nvrtcVersion(ctypes.byref(maj), ctypes.byref(min_))
print("NVRTC path:", nvrtc_path)
print("NVRTC version:", f"{maj.value}.{min_.value}")   # should print 12.0


NVRTC path: /home/dwong/anaconda3/lib/libnvrtc.so.12
NVRTC version: 12.0


In [10]:
from numba import cuda
print("CUDA available:", cuda.is_available())

from numba.cuda.cudadrv import driver
print("Driver version:", driver.get_version())  # expect (12, 0) or similar


CUDA available: True
Driver version: (12, 0)


In [11]:
import numpy as np

@cuda.jit
def add1(x):
    i = cuda.grid(1)
    if i < x.size:
        x[i] += 1

a = np.zeros(128, np.float32)
d = cuda.to_device(a)
add1[1, 128](d)
print(d.copy_to_host()[:8])  # expect ones


AssertionError: key already in dictionary: <class 'numba.core.target_extension.CUDA'>

In [6]:
import os
os.environ["LD_LIBRARY_PATH"] = "/home/dwong/anaconda3/lib" + (":" + os.environ["LD_LIBRARY_PATH"] if os.environ.get("LD_LIBRARY_PATH") else "")
